# Async Multi-Node RL Post-Training with GRPO on Union — from Google Colab

This notebook runs **multi-node reinforcement learning for LLMs** (GRPO + LoRA — the family of
techniques behind reasoning models like DeepSeek-R1) **on your Union cluster, driven from Colab**.
It is Part 2 of the GRPO tutorial: the single-GPU loop from
[Part 1](https://github.com/unionai/unionai-examples/tree/main/v2/tutorials/rl_grpo_lora) is
imported here as a library (its warm vLLM pool, reward, and eval), and the trainer becomes a
**gang of pods** on a `ClusteredTaskEnvironment`, with generation pipelined against training.

Colab is only the *client*: nothing heavy runs in this notebook. Each cell below submits work to
Union, which provisions the GPUs, runs the tasks, and streams a live report you can watch in the
browser.

```
  Colab (this notebook)                    Your Union cluster
  ─────────────────────                    ──────────────────────────────────────────────
  pip install flyte         ──submit──▶    driver task (the async RL loop, CPU)
  authenticate (API key)                     ├─ warm vLLM pool        (1 GPU × N replicas) ← rollouts + held-out eval
  run(train_rl_clustered)                    ├─ reward tasks          (CPU)                ← verifiable scoring
  watch run.url                              └─ train_step_clustered  (JobSet: R pods × G GPUs, torchrun/DDP)
                                                                                           ← real GRPO step, overlapped with
                                                                                             the NEXT batch's generation
```

### What you need
1. A **Union deployment with GPU capacity** — single-GPU nodes for the pool, and multi-GPU nodes
   (e.g. `g6e.12xlarge`, 4× L40S) for the stress run.
2. A **Union API key** — a single base64 string that encodes the endpoint, org, and credentials,
   so no `config.yaml` or browser login is needed. Create one from the **Union web console**
   (org/admin settings → API keys / apps), or with the `union` CLI from a machine where you're
   already logged in. *(Note: the v2 `flyte` CLI does not create API keys.)*
3. A **Hugging Face token** stored on the cluster as a secret named `hf-token`
   (one-time setup — see the secret cell below).

The flow: set up once → run the **`smoke`** profile (~25 min, proves everything) → switch to the
**`stress`** profile (8B policy on GSM8K, 2 pods × 4 GPUs) → optionally the sequential comparison.
The profile is a single environment variable, `GRPO_PROFILE`.


## 1. Install the client

Only the Flyte SDK and two tiny pure-Python deps — **no torch, no vLLM, no CUDA** in Colab.
The tutorial module imports GPU libraries lazily *inside* the tasks, which run on the cluster.

In [ ]:
%pip install -q "flyte>=2.4.0" "async-lru>=2.0.0" "pydantic>=2"

## 2. Get the tutorial code

The tutorial is three modules — `rl_grpo_lora.py` (Part 1: envs, warm vLLM pool, reward, eval),
`report_helpers.py` (the live report), and `rl_grpo_lora_clustered.py` (Part 2: the clustered
trainer + async driver). Upload your local copies; Flyte bundles the folder and ships it to the
cluster when a run is submitted.


In [ ]:
import pathlib, sys
from google.colab import files

TUTORIAL_DIR = pathlib.Path("/content/rl_grpo")
TUTORIAL_DIR.mkdir(exist_ok=True)

NEEDED = {"rl_grpo_lora.py", "report_helpers.py", "rl_grpo_lora_clustered.py"}
missing = NEEDED - {p.name for p in TUTORIAL_DIR.iterdir()}
if missing:
    print("Select:", ", ".join(sorted(missing)))
    for name, data in files.upload().items():  # pick the three tutorial modules
        (TUTORIAL_DIR / name).write_bytes(data)
        print("wrote", TUTORIAL_DIR / name)

assert not (NEEDED - {p.name for p in TUTORIAL_DIR.iterdir()}), "still missing modules"
sys.path.insert(0, str(TUTORIAL_DIR))


### Pick the profile and GPU type

Both are read **at import time** and baked into every task's spec (and re-injected into each
environment as env vars, so the containers resolve identical constants):

- `GRPO_PROFILE`
  - `smoke` (default) — Qwen3-0.6B, the toy arithmetic set, 3 iterations, 2 single-GPU trainer
    pods. ~25 min. Run this first: it exercises every path (pool, reward, clustered JobSet,
    real GRPO objective, eval, report) cheaply.
  - `stress` — Qwen3-8B, GSM8K, 20 iterations, 32 prompts × 6 samples per iteration, 2 trainer
    pods × **4 L40S** each (world size 8). Needs multi-GPU nodes. This is the configuration the
    tutorial's async claim is actually about: with 192 rollouts of up to 512 tokens per
    iteration, *generation* is the long pole, so the clustered trainer's whole lifetime hides
    under it.
- `GRPO_GPU_TYPE` — on this tenant L4 nodes wedge (`NotReady`), so everything runs on L40S.

Start with `smoke`. Section 8 has a cell that switches to `stress` **in place** (it reloads the
modules and re-prefetches the base model), so no runtime restart is needed.


In [ ]:
import os

os.environ.setdefault("GRPO_PROFILE", "smoke")   # "stress" for the real run — see above
os.environ.setdefault("GRPO_GPU_TYPE", "L40s")   # set BEFORE importing any tutorial module
print("profile:", os.environ["GRPO_PROFILE"], "| GPU type:", os.environ["GRPO_GPU_TYPE"])


## 3. Authenticate to your Union cluster

Paste your API key when prompted (it is not stored in the notebook). Three things worth noting:

- `init_from_api_key` is the headless auth path — no browser redirect, which is exactly what a
  Colab runtime needs. (Locally you'd normally use `flyte.init_from_config()` with a
  `config.yaml` and a browser login; that flow doesn't fit a hosted notebook.)
- `image_builder="remote"` makes Union build the task container image **on the cluster** the
  first time (vLLM + flashinfer + PEFT — takes a few minutes once, cached afterwards). Colab has
  no Docker, so the local builder would fail.
- `root_dir` points at the tutorial folder so the code bundle contains all three modules.

In [ ]:
import getpass
import flyte

api_key = getpass.getpass("Union API key: ")

flyte.init_from_api_key(
    api_key=api_key,
    project="default",          # ← change to your project
    domain="development",       # ← change to your domain
    root_dir=TUTORIAL_DIR,
    image_builder="remote",
)
print("Authenticated.")

## 4. One-time: the Hugging Face token secret

The rollout and trainer tasks read a cluster secret named `hf-token` to pull the base model.
If your cluster already has it, skip this cell. Creating it is a one-time step
(`flyte create secret NAME --value ...`):

In [ ]:
# Skip if the `hf-token` secret already exists on the cluster.
hf_token = getpass.getpass("Hugging Face token (or blank to skip): ")
if hf_token:
    import subprocess
    subprocess.run(["flyte", "create", "secret", "hf-token", "--value", hf_token], check=True)
    print("Secret created.")
else:
    print("Skipped — assuming hf-token already exists.")

## 5. Prefetch the base model (once per profile)

Rather than every task pulling from Hugging Face, Union downloads the profile's base model
(**Qwen3-0.6B** for `smoke`, **Qwen3-8B** for `stress`) once into the cluster's object store; the
resulting directory is handed to every task as a `flyte.io.Dir`. The helper is reused by the
profile-switch cell in section 8.


In [ ]:
import inspect, asyncio
import flyte.prefetch
import rl_grpo_lora as tutorial


def prefetch_base(repo: str):
    """Download `repo` from Hugging Face into the object store once; return its flyte.io.Dir."""
    run = flyte.prefetch.hf_model(repo=repo, hf_token_key="hf-token")
    print("Prefetch run:", run.url)
    run.wait()
    outs = run.outputs()
    if inspect.isawaitable(outs):
        outs = asyncio.get_event_loop().run_until_complete(outs)
    return outs[0]


base_dir = prefetch_base(tutorial.BASE_MODEL_REPO)   # follows GRPO_PROFILE
print("Base model in object store:", base_dir.path)


## How the multi-node loop works

In Part 1 the GRPO gradient step ran on **one GPU in one pod**. When the policy outgrows a
single GPU (or you want bigger rollout batches per step), the trainer needs to become a
**gang of pods training cooperatively** — multiple nodes, one `torch.distributed` process
group, starting together and failing together.

That is exactly what a `ClusteredTaskEnvironment` declares. Each call to a task in this
environment launches **one Kubernetes JobSet**: `replicas` pods, each bootstrapped by
`torchrun` with `nproc_per_node` workers, all rendezvoused into a single world. You write one
Python function; it runs on every rank; rank 0's return value is the task's output.

```python
from flyte.clustered import ClusteredTaskEnvironment, ClusterFailurePolicy, TorchRun

train_env = ClusteredTaskEnvironment(
    name="rl-grpo-train-cluster",
    image=image,                                   # same image as Part 1
    resources=flyte.Resources(gpu=flyte.GPU("L40s", 4), cpu=16, memory="120Gi"),  # PER POD
    replicas=2,                                    # 2 pods == 2 nodes
    nproc_per_node=4,                              # ranks per pod; world_size = 8
    runtime=TorchRun(),                            # in-pod torchrun bootstrap
    failure_policy=ClusterFailurePolicy(max_restarts=2),  # any pod dies → whole gang restarts
)
```

Three more ideas ride on top of the environment change:

- **The real GRPO objective.** The trainer computes token-level importance ratios from vLLM's
  sampling-time logprobs (carried in each `Rollout`), clips them PPO-style, and adds a KL
  penalty against the adapter-disabled base. This is what makes training on *stale* rollouts
  sound — which the next point exploits. AdamW state persists across iterations through the
  object store, like the adapter.
- **Async pipelining** (`pipelined=True`): while the clustered trainer spins up and trains on
  batch N, the warm pool is already generating batch N+1 with the one-step-stale adapter — the
  JobSet's whole lifetime, cold start included, disappears under generation. `pipelined=False`
  runs the identical loop sequentially so the timing chart can prove the difference.
- **A held-out eval**, off the critical path: greedy decoding on a fixed question set, once for
  the raw base model (the baseline) and after every step. Each eval runs in the background and
  overlaps the *next* train step; only the final one is waited on.

One DDP discipline the multi-node trainer has to respect, and the tutorial code shows how:
**every rank must execute the same number of collective operations.** Per-sample `backward()`
with data-dependent skips desyncs NCCL the moment ranks disagree; see `ddp_shard_step` in
`rl_grpo_lora_clustered.py` for the `no_sync` pattern that makes the invariant hold by
construction, empty shards included.


## 6. Launch the multi-node loop 🚀

This submits the **driver task** and returns immediately with a URL. Remotely, the driver runs:

> baseline eval → [ sample prompts → generate on the **warm pool** → **score** ] ‖ [ **clustered
> GRPO step** on batch N ] → eval in the background → repeat

The cell uses whatever profile is active — `smoke` the first time through, `stress` after the
switch in section 8.

**Note the `with_runcontext(interactive_mode=False)`.** In a notebook the SDK defaults to
"interactive mode", which ships the task as a *cloudpickle only* — no source files. Because the
tutorial's modules import each other, the container would unpickle the task, try to import
`report_helpers`, and fail with `ModuleNotFoundError`. Turning interactive mode off makes the
SDK build a normal source bundle from `root_dir` instead, shipping every loaded module.

**Open the printed URL.** In the DAG you'll see an `eval-baseline` group, then per iteration an
`iter-N-generate` group (warm-pool actor tasks), an `iter-N-train` group whose single node fans
out into a **JobSet of `replicas` pods**, and an `iter-N-eval` group. The *Report* tab updates
after every iteration.


In [ ]:
import rl_grpo_lora_clustered as part2

# Dataset, iterations, model, and trainer shape all follow GRPO_PROFILE; override here to deviate.
print(f"profile={tutorial.PROFILE_NAME}: {tutorial.BASE_MODEL_REPO}, {tutorial.DEFAULT_DATASET}, "
      f"{part2.NUM_ITERATIONS} iters, trainer {part2.REPLICAS}x{part2.NPROC_PER_NODE} GPUs")
rl2_run = flyte.with_runcontext(interactive_mode=False).run(
    part2.train_rl_clustered,
    base=base_dir,                       # the prefetched base model for this profile
    num_iterations=part2.NUM_ITERATIONS, # profile default (smoke: 3, stress: 20)
    dataset=tutorial.DEFAULT_DATASET,    # profile default (smoke: arithmetic, stress: gsm8k)
    pipelined=True,
)
print("▶ Watch the multi-node run:", rl2_run.url)


## 7. Wait for completion, fetch the adapter, check the receipts

You can leave this running — even if Colab disconnects, **the run continues on the cluster**
(that's the point of Colab-as-client). Re-attach later from any machine, or just use the UI.

When it finishes, the Report tab should show, even on the smoke profile:

- **Per-iteration table**: `contributing` > 0 — the gradient path really ran on every rank.
- **Off-policy drift chart**: importance ratio ≈ 1, clip fraction well under 1%, KL exactly 0 on
  step 1 (a fresh LoRA *is* the base model) and small-but-nonzero afterwards. The ~0.5% clip
  fraction you see even fully on-policy is pure vLLM-vs-HuggingFace numerics mismatch — a known
  training/inference gap, measured here for free.
- **Timing chart**: on the smoke profile the clustered step (~2–10 min including JobSet
  spin-up) is *longer* than the ~1 min of generation, so pipelining can only hide that minute.
  That inversion is the point of the stress profile below.
- **Held-out eval**: on the toy set it reuses the 8 training questions — a wiring check only.


In [ ]:
rl2_run.wait()

outputs = rl2_run.outputs()
if inspect.isawaitable(outputs):
    outputs = asyncio.get_event_loop().run_until_complete(outputs)
if not outputs or outputs[0] is None:
    raise RuntimeError(
        f"No outputs — the run did not succeed. Open {rl2_run.url}, click the failed action, "
        "and read its logs/error before re-running."
    )
adapter = outputs[0]
print("Trained LoRA adapter (flyte.io.Dir):", adapter.path)


## 8. Switch to the stress profile — where the async claim becomes true

The cell below flips `GRPO_PROFILE`, reloads the three modules (they read the profile at import
time, so a plain env-var change is not enough), and prefetches **Qwen3-8B**. Then go back and
**re-run sections 6 and 7**: the pool replicas size up for the 8B, and the trainer becomes
**2 pods × 4 L40S** (world size 8, two clipped epochs per step) on 192 rollouts of up to 512
GSM8K tokens per iteration. To go back, set the value to `"smoke"` and run the cell again.

What to expect, from a reference run on a Union demo tenant:

- **Held-out GSM8K accuracy (64 questions, greedy): 61% baseline → ~86–92% after 20 steps.**
  Part of that gain is the model learning the task's answer format and length budget under a
  deliberately plain prompt (no chat template, 512-token cap) — legitimate RLVR behaviour, and
  worth stating plainly rather than reading it as new math ability.
- **Steady state ≈ 5 min per iteration, all of it generation.** The clustered step
  (JobSet spin-up + 8B load + training, ~2.3 min) starts 1–2 s after the next batch's
  generation begins and finishes ~2 min *before* it — entirely hidden. Sequentially the same
  iteration is ~7.3 min, so pipelining is worth roughly a third of the wall-clock.
- **Evals never touch the critical path**; only the final one is waited on.
- **`contributing` falls from ~180 to ~100 of 192** as more prompt groups become all-correct —
  textbook GRPO saturation, and the cue to move to harder prompts.
- **The first JobSet can wait a long time for 4-GPU nodes.** On the reference run Karpenter
  took 3½ hours to obtain two `g6e.12xlarge` instances; the driver simply waited (iteration-1
  generation had already finished) and the run completed without a failure. Budget for it, and
  give the tenant owner a heads-up before pinning two 4-GPU nodes for a couple of hours.


In [ ]:
import importlib, os
import report_helpers, rl_grpo_lora, rl_grpo_lora_clustered

os.environ["GRPO_PROFILE"] = "stress"   # or "smoke" to switch back

# Reload in dependency order so every module sees the new profile.
for m in (report_helpers, rl_grpo_lora, rl_grpo_lora_clustered):
    importlib.reload(m)
tutorial, part2 = rl_grpo_lora, rl_grpo_lora_clustered
print(f"profile={tutorial.PROFILE_NAME}: {tutorial.BASE_MODEL_REPO}, {tutorial.DEFAULT_DATASET}, "
      f"{part2.NUM_ITERATIONS} iters, trainer {part2.REPLICAS}x{part2.NPROC_PER_NODE} GPUs")

base_dir = prefetch_base(tutorial.BASE_MODEL_REPO)   # the profile's base model (cached if already fetched)
print("Base model in object store:", base_dir.path)
print("→ now re-run sections 6 and 7")


## 9. Optional: the sequential comparison

The timing chart's overlap figure (`Σ generate + train − iteration`) is measured within a single
run, but the cleanest evidence is a second run with pipelining off — same objective, same
trainer, same eval, only the overlap removed. Run it under the same profile and compare the two
Report tabs.


In [ ]:
rl2_seq = flyte.with_runcontext(interactive_mode=False).run(
    part2.train_rl_clustered,
    base=base_dir,
    num_iterations=part2.NUM_ITERATIONS,
    dataset=tutorial.DEFAULT_DATASET,
    pipelined=False,                     # generate → train → generate → train ...
)
print("▶ Sequential run:", rl2_seq.url)


## Appendix (optional): the single-GPU Part-1 loop

Nothing above depends on running Part 1 — Part 2 only *imports* it. Run this if you want to see
the single-pod loop from the original tutorial in action. It is pinned to 3 iterations: under
the stress profile it would otherwise spend hours doing 20 iterations of the simplified
REINFORCE objective on one GPU. Note it is **not** the right baseline for the pipelining claim
(different objective, one optimizer step per iteration, no eval); section 9 is.


In [ ]:
rl1_run = flyte.with_runcontext(interactive_mode=False).run(
    tutorial.train_rl,
    base=base_dir,
    num_iterations=3,   # pinned: this is a demonstration, not a benchmark
)
print("▶ Part-1 single-GPU run:", rl1_run.url)
